
# (Re-)Imag(in)ing Price Trends — unified notebook (Mac M4 · Kaggle 2×T4)

One notebook, both machines. It **auto-detects** the device and sizes the parallelism to it:

| | Mac M4 (`mps`) | Kaggle 2×T4 (`cuda`) |
|---|---|---|
| Image generation | all performance cores | (skipped — dataset uploaded) |
| Training | 1 model at a time (Metal parallelises internally) | `MODELS_PER_GPU × 2` models **concurrently**, round-robined across both GPUs |
| Loader workers | auto | auto-capped so `MAX_PARALLEL × NUM_WORKERS ≤ vCPU` |

**Resumable across machines.** Every ensemble member's test predictions are checkpointed to `PRED_DIR` as a `.npy`. Train 2 members on the Mac, copy the `.npy` files over, train the other 3 on Kaggle — the notebook assembles the ensemble from whatever it finds and skips anything already done. Config-level results are appended to a CSV that is also resumed.

**Correctness fixes baked in** (see §3):
- the blank-image indexing bug is fixed at generation *and* filtered at load, so **old, buggy datasets still work**;
- `SIGNAL_GATE=False` makes the CNN's sample set depend only on the image (paper-faithful);
- `BAGGING=False` — the paper's fixed-split ensemble;
- non-overlapping decile portfolios (JKX Table 3).

**Memory:** models, loaders and worker processes are explicitly torn down between members, and RSS + GPU memory are printed every epoch so leaks are visible, not inferred.

In [29]:
import os, gc, sys, time, json, tempfile, threading, platform
import multiprocessing as mp
try: mp.set_start_method('fork', force=True)     # fork lets render workers inherit data copy-on-write
except RuntimeError: pass
import concurrent.futures as cf
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

try:
    import psutil; _PROC = psutil.Process()
except ImportError:
    _PROC = None

# ---------------------------------------------------------------- device
ON_KAGGLE = os.path.isdir('/kaggle')
if torch.cuda.is_available():
    BACKEND = 'cuda'
    DEVICES = [f'cuda:{i}' for i in range(torch.cuda.device_count())]
    torch.backends.cudnn.benchmark = True          # fixed input shapes per config
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    BACKEND, DEVICES = 'mps', ['mps']
else:
    BACKEND, DEVICES = 'cpu', ['cpu']

CPU = mp.cpu_count()

# ---------------------------------------------------------------- knobs
TRAIN_END  = '2004-12-31'   # in-sample block
OOS_START  = '2005-01-01'
OOS_END    = '2026-12-31'   # extended: the parquet now runs to 2026

BATCH_SIZE = 128            # paper
EPOCHS     = 100
LR         = 1e-5
PATIENCE   = 2
USE_AMP    = False          # False = fp32 (paper-exact). True = fp16 autocast on CUDA (~1.5-2x, slightly different numerics)

# concurrency: several whole models at once on CUDA; Metal prefers one model with internal parallelism
MODELS_PER_GPU = 2
if BACKEND == 'cuda':
    MAX_PARALLEL = max(1, len(DEVICES) * MODELS_PER_GPU)
    NUM_WORKERS  = max(0, min(2, CPU // MAX_PARALLEL))     # never oversubscribe the CPU
else:
    MAX_PARALLEL = 1
    NUM_WORKERS  = min(4, max(0, CPU - 2))

# ensemble / methodology
ENSEMBLE_SIZE     = 5
BAGGING           = False   # paper-faithful: one fixed split, diversity from random init
TRAIN_FRAC        = 0.7
SHARED_SPLIT_SEED = 42
SIGNAL_GATE       = False   # False = CNN samples gated by the IMAGE only (paper). True = also require MOM/STR/WSTR/TREND
                            # (the old, buggy behaviour). Changing this CHANGES the dataset -> regenerate.
DROP_INCOMPLETE   = True    # drop windows without a full lookback. No-op on a clean dataset; RESCUES an old buggy one.

# which configs to run in THIS session (lets you split work across machines)
IMAGE_SIZES = [5, 20, 60]
HORIZONS    = [5, 20, 60]
TESTING     = False         # True -> 3 tickers, 1 config, 2 members

# ---------------------------------------------------------------- paths
if ON_KAGGLE:
    IMG_DIR      = '/kaggle/input/price-trend-images'     # <-- EDIT: uploaded image dataset (read-only)
    PARQUET_PATH = '/kaggle/input/market-data-sp500/market_data_sp500_1997_2026.parquet'
    OUT_DIR      = '/kaggle/working'
else:
    IMG_DIR      = './image_store'                        # generated here on the Mac
    PARQUET_PATH = './market_data_sp500_1997_2026.parquet'
    OUT_DIR      = '.'
PRED_DIR    = os.path.join(OUT_DIR, 'preds');   os.makedirs(PRED_DIR, exist_ok=True)
RESULTS_CSV = os.path.join(OUT_DIR, 'results.csv')
os.makedirs(IMG_DIR, exist_ok=True) if not ON_KAGGLE else None

# ---------------------------------------------------------------- logging (thread-safe: concurrent models interleave)
_LOCK = threading.Lock()
_T0   = time.time()

def log(msg, tag=''):
    el = time.time() - _T0
    with _LOCK:
        print(f"[{el//3600:02.0f}:{(el%3600)//60:02.0f}:{el%60:04.1f}]{(' '+tag) if tag else ''} {msg}", flush=True)

def mem_str(dev=None):
    parts = []
    if _PROC is not None:
        parts.append(f"rss {_PROC.memory_info().rss/1e9:.1f}G")
    if BACKEND == 'cuda' and dev is not None:
        i = dev.index or 0
        parts.append(f"gpu{i} {torch.cuda.memory_allocated(i)/1e9:.1f}/{torch.cuda.memory_reserved(i)/1e9:.1f}G")
    elif BACKEND == 'mps':
        try: parts.append(f"mps {torch.mps.current_allocated_memory()/1e9:.1f}G")
        except Exception: pass
    return " ".join(parts)

def free_memory(dev=None):
    """Called between ensemble members and configs. The single most important anti-leak step."""
    gc.collect()
    if BACKEND == 'cuda':
        if dev is not None and dev.type == 'cuda':
            with torch.cuda.device(dev): torch.cuda.empty_cache()
        else:
            for i in range(len(DEVICES)): 
                with torch.cuda.device(i): torch.cuda.empty_cache()
    elif BACKEND == 'mps':
        try: torch.mps.empty_cache()
        except Exception: pass

log(f"host={'kaggle' if ON_KAGGLE else platform.node()} backend={BACKEND} devices={DEVICES} vCPU={CPU}")
log(f"parallel models={MAX_PARALLEL} workers/model={NUM_WORKERS} (loader procs {MAX_PARALLEL*NUM_WORKERS}/{CPU})")
log(f"IMG_DIR={IMG_DIR}")
log(f"OUT_DIR={OUT_DIR} (preds -> {PRED_DIR})")
log(f"SIGNAL_GATE={SIGNAL_GATE} BAGGING={BAGGING} DROP_INCOMPLETE={DROP_INCOMPLETE} AMP={USE_AMP} mem={mem_str()}")

[00:00:00.0] host=Yashs-MacBook-Pro.local backend=mps devices=['mps'] vCPU=10
[00:00:00.0] parallel models=1 workers/model=4 (loader procs 4/10)
[00:00:00.0] IMG_DIR=./image_store
[00:00:00.0] OUT_DIR=. (preds -> ./preds)
[00:00:00.0] SIGNAL_GATE=False BAGGING=False DROP_INCOMPLETE=True AMP=False mem=rss 1.0G mps 0.0G


## 1. Market data

In [30]:
if not os.path.exists(PARQUET_PATH):
    raise FileNotFoundError(f"parquet not found: {PARQUET_PATH}")
df_market = pd.read_parquet(PARQUET_PATH)
df_market['Date'] = pd.to_datetime(df_market['Date'])

if TESTING:
    keep = df_market['Ticker'].unique()[:3]
    df_market = df_market[df_market['Ticker'].isin(keep)].copy()
    IMAGE_SIZES, HORIZONS, ENSEMBLE_SIZE = [5], [5], 2
    log("TESTING MODE: 3 tickers, 1 config, 2 members")

log(f"parquet: {len(df_market):,} rows | {df_market['Ticker'].nunique()} tickers | "
    f"{df_market['Date'].min().date()} -> {df_market['Date'].max().date()} | mem={mem_str()}")

[00:00:00.8] parquet: 4,350,152 rows | 748 tickers | 1997-01-02 -> 2026-06-30 | mem=rss 1.6G mps 0.0G


## 2. Image generation — exact PIL rendering (verified against the paper)

In [31]:
def _height(n_days): return 32 if n_days == 5 else (64 if n_days == 20 else 96)
def img_shape(n_days): return _height(n_days), n_days * 3

def generate_image_uint8(window_df, n_days):
    height, width = img_shape(n_days)
    img = Image.new('L', (width, height), color=0)
    if len(window_df) == 0:
        return np.zeros((height, width), dtype=np.uint8)
    first_close = window_df['Close'].iloc[0]
    if first_close == 0 or np.isnan(first_close):
        return np.zeros((height, width), dtype=np.uint8)
    draw = ImageDraw.Draw(img)
    prices  = window_df[['Open','High','Low','Close']].values / first_close
    ma      = window_df['Close'].rolling(window=n_days, min_periods=1).mean().values / first_close
    volumes = window_df['Volume'].values
    max_p, min_p = np.nanmax(prices), np.nanmin(prices); max_v = np.nanmax(volumes)
    if max_p == min_p: max_p = min_p + 1e-5
    if max_v == 0:     max_v = 1e-5
    price_h = int(height * 0.8); vol_h = height - price_h
    def p2y(p): return int(np.clip(price_h-1-(p-min_p)/(max_p-min_p)*(price_h-1), 0, price_h-1))
    def v2y(v): return int(np.clip(height-1-(v/max_v)*(vol_h-1), price_h, height-1))
    ma_pts = []
    for i in range(len(window_df)):
        x = i*3 + 1
        oy, hy, ly, cy = p2y(prices[i,0]), p2y(prices[i,1]), p2y(prices[i,2]), p2y(prices[i,3])
        draw.line([(x,hy), (x,ly)], fill=255, width=1)
        draw.line([(x-1,oy), (x,oy)], fill=255, width=1)
        draw.line([(x,cy), (x+1,cy)], fill=255, width=1)
        draw.line([(x,height-1), (x,v2y(volumes[i]))], fill=255, width=1)
        ma_pts.append((x, p2y(ma[i])))
    if len(ma_pts) > 1:
        draw.line(ma_pts, fill=255, width=1)
    return np.array(img, dtype=np.uint8)

## 3. Image store — build (Mac) / load (Kaggle)

**The window-validity mask, fixed.** The old mask was
`t_df[REQUIRED].isna().rolling(n).sum().sum(axis=1) == 0`.
For the first `n-1` rows the rolling window is incomplete → every column is `NaN` → pandas' `sum(axis=1)` turns an all-`NaN` row into `0.0`, which **passes**. Those rows then sliced `iloc[idx-n+1 : idx+1]` with a *negative* start (`iloc[-19:1]`) → an **empty window** → an all-black image paired with a real label. Every ticker emitted `n-1` of them.

The fix reduces NaN to a per-day flag *first*, then rolls: an incomplete leading window stays `NaN`, and `NaN == 0` is `False`, so it is correctly excluded.

```python
bad_day    = t_df[COLS].isna().any(axis=1).astype(float)
window_bad = bad_day.rolling(n_days).sum()      # NaN on incomplete leading windows
valid      = np.where(window_bad.values == 0)[0]
```

`DROP_INCOMPLETE=True` *also* filters `end_idx < n_days-1` at load time — a no-op on a freshly built store, but it **rescues an old buggy dataset without regenerating**, since every corrupted sample is exactly that set.

In [32]:
IMAGE_COLS  = ['Open','High','Low','Close','Volume']            # what the image needs (paper: CNN sees only this)
MA_LAGS     = [3, 5, 10, 20, 50, 100, 200]
MA_COLS     = [f'MA{L}' for L in MA_LAGS]                       # raw inputs to the Han-Zhou-Zhu trend factor
SIGNAL_COLS = ['MOM','STR','WSTR'] + MA_COLS                    # benchmarks, NOT CNN inputs
META_COLS   = SIGNAL_COLS

def _index_ticker(args):
    ticker, t_df, n_days, gate = args
    t_df = t_df.sort_values('Date').reset_index(drop=True)
    cols = IMAGE_COLS + (SIGNAL_COLS if gate else [])
    bad_day    = t_df[cols].isna().any(axis=1).astype(float)
    window_bad = bad_day.rolling(window=n_days).sum()            # NaN on the first n_days-1 rows
    valid      = np.where(window_bad.values == 0)[0]             # -> leading rows correctly excluded
    if 'InIndex' in t_df.columns:                                # sample ONLY on dates the stock was an
        valid = valid[t_df['InIndex'].values[valid].astype(bool)] # S&P 500 member (kills survivorship bias);
                                                                  # lookback prices from non-member days are
                                                                  # still real prices and remain usable.
    sig = {c: t_df[c].values for c in META_COLS}
    recs = [dict({'ticker': ticker, 'end_idx': int(i), 'end_date': str(t_df['Date'].iloc[i].date())},
                 **{c: float(sig[c][i]) for c in META_COLS})
            for i in valid]
    return ticker, t_df, recs

_G = {}
def _render_chunk(rng):
    lo, hi = rng
    mm = np.memmap(_G['dat'], dtype=np.uint8, mode='r+', shape=_G['shape'])
    nd = _G['n_days']
    for i in range(lo, hi):
        s = _G['recs'][i]; tdf = _G['grouped'][s['ticker']]; e = s['end_idx']
        mm[i, 0] = generate_image_uint8(tdf.iloc[e-nd+1 : e+1], nd)
    mm.flush(); del mm
    return hi - lo

def store_paths(n_days):
    return (os.path.join(IMG_DIR, f'raw_n{n_days}.dat'),
            os.path.join(IMG_DIR, f'meta_n{n_days}.parquet'))

def build_store(df, n_days):
    """Render every valid window once into a raw uint8 memmap. Skips if already present."""
    dat, meta = store_paths(n_days)
    H, W = img_shape(n_days)
    if os.path.exists(dat) and os.path.exists(meta):
        log(f"N={n_days}: store already present, skipping generation.")
        return
    t0 = time.time()
    log(f"N={n_days}: generating images (gate_signals={SIGNAL_GATE})...")

    args = [(t, g.copy(), n_days, SIGNAL_GATE) for t, g in df.groupby('Ticker')]
    grouped, recs = {}, []
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        for tk, tdf, r in tqdm(ex.map(_index_ticker, args), total=len(args),
                               desc=f"index N={n_days}", leave=False):
            grouped[tk] = tdf; recs.extend(r)
    N = len(recs)
    if N == 0:
        log(f"N={n_days}: no valid windows."); return
    gb = N*H*W/1e9
    log(f"N={n_days}: {N:,} windows -> {gb:.1f} GB raw ({H}x{W} px). rendering on {CPU} cores...")

    np.memmap(dat, dtype=np.uint8, mode='w+', shape=(N,1,H,W)).flush()      # preallocate
    m = pd.DataFrame(recs)
    m.attrs = {}
    m.to_parquet(meta)

    _G.update(dat=dat, shape=(N,1,H,W), n_days=n_days, grouped=grouped, recs=recs)
    chunk  = max(1, N // (CPU*8))
    ranges = [(i, min(i+chunk, N)) for i in range(0, N, chunk)]
    with cf.ProcessPoolExecutor(max_workers=CPU) as ex:
        list(tqdm(ex.map(_render_chunk, ranges), total=len(ranges),
                  desc=f"render N={n_days}", leave=False))
    _G.clear(); del grouped, recs, m
    free_memory()
    log(f"N={n_days}: built in {(time.time()-t0)/60:.1f} min -> {dat} | mem={mem_str()}")

def load_store(n_days, df=None):
    """Memory-light loader. Keeps only the small metadata table.
 
    If `df` is given, every row is validated against it: rows whose (ticker, end_date)
    no longer sits at the recorded `end_idx` were rendered from different data and are
    dropped. This makes the store robust to a parquet that changed after generation.
    """
    dat, meta = store_paths(n_days)
    if not (os.path.exists(dat) and os.path.exists(meta)):
        return None
    m = pd.read_parquet(meta).reset_index(drop=True)
    H, W = img_shape(n_days)
    N = len(m)
    expect, actual = N * H * W, os.path.getsize(dat)
    if actual != expect:
        raise RuntimeError(f"N={n_days}: .dat is {actual:,} bytes but metadata implies {expect:,} "
                           f"({N:,} imgs of {H}x{W}). Wrong dataset entirely.")
    m['_row'] = np.arange(N)                       # absolute row in the .dat -- fixed forever
 
    if DROP_INCOMPLETE:                            # the old blank-image indexing bug
        bad = m['end_idx'].values < (n_days - 1)
        if bad.any():
            log(f"N={n_days}: dropping {int(bad.sum()):,} blank rows (incomplete lookback).")
            m = m[~bad].reset_index(drop=True)
 
    if df is not None:                             # <-- the rescue
        ok, _ = align_store(m, df, verbose=False)
        n_bad = int((~ok).sum())
        if n_bad:
            log(f"N={n_days}: dropping {n_bad:,} rows ({100*(1-ok.mean()):.2f}%) whose images no "
                f"longer match the parquet (rendered before it changed). .dat untouched.")
            m = m[ok].reset_index(drop=True)
        if len(m) == 0:
            raise RuntimeError(f"N={n_days}: NO rows survive alignment. The parquet is not the one "
                               f"this store was rendered from. Re-render required.")
 
    log(f"N={n_days}: {N:,} images on disk ({actual/1e9:.1f} GB), {len(m):,} usable | mem={mem_str()}")
    return {'dat': dat, 'shape': (N, 1, H, W), 'meta': m}

## 4. Labels, splits, dataset

Images are horizon-independent, so one store serves all three horizons — only the label changes. The **test set is seed-independent and chronological**, so it is identical for every ensemble member; that is what makes averaging their predictions column-wise valid (and what lets you train members on *different machines* and still combine them).

In [33]:
def label_for_horizon(store, df, horizon):
    m = store['meta']
    closes = {t: g.sort_values('Date')['Close'].values.astype(np.float64)   # small: no DataFrames retained
              for t, g in df.groupby('Ticker')}
    fwd = np.full(len(m), np.nan)
    for t, pos in m.groupby('ticker').indices.items():
        c = closes.get(t)
        if c is None: continue
        ei = m['end_idx'].values[pos]
        ok = (ei + horizon) < len(c)
        base_p = c[ei]
        fwd_p  = np.where(ok, c[np.minimum(ei + horizon, len(c)-1)], np.nan)
        with np.errstate(invalid='ignore', divide='ignore'):
            fwd[pos] = np.where(ok & (base_p != 0), fwd_p/base_p - 1.0, np.nan)
    del closes

    d = m.copy(); d['Fwd_Ret'] = fwd
    d = d[~np.isnan(d['Fwd_Ret'].values)].reset_index(drop=True)   # drop the undefined-label tail
    rows  = d['_row'].values                                        # absolute rows into the .dat
    label = (d['Fwd_Ret'].values > 0).astype(np.int64)
    base  = d.rename(columns={'ticker':'Ticker', 'end_date':'Date'})[
                ['Date','Ticker'] + META_COLS + ['Fwd_Ret']]
    return rows, label, base

def trainval_positions(base):
    d = base['Date'].values
    return np.where(d <= TRAIN_END)[0]

def test_positions(base):
    d = base['Date'].values                                        # seed-independent AND chronological
    te = np.where((d >= OOS_START) & (d <= OOS_END))[0]
    return te[np.argsort(d[te], kind='stable')]

def subsample_split(tv, seed, frac=TRAIN_FRAC):
    tv = tv.copy()
    np.random.default_rng(seed).shuffle(tv)
    cut = int(frac * len(tv))
    return tv[:cut], tv[cut:]

class ImgDataset(Dataset):
    def __init__(self, dat, shape, rows, labels=None):
        self.dat=dat; self.shape=tuple(int(x) for x in shape)
        self.rows=np.asarray(rows); self.labels=labels; self._mm=None
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        if self._mm is None:                                       # mapped lazily: one handle per worker
            self._mm = np.memmap(self.dat, dtype=np.uint8, mode='r', shape=self.shape)
        img = torch.from_numpy(np.ascontiguousarray(self._mm[int(self.rows[i])]))   # (1,H,W) uint8
        if self.labels is None: return img
        return img, int(self.labels[i])
    def close(self):
        self._mm = None                                            # release the mapping

## 5. CNN — verbatim

In [34]:
def xavier_init(m):
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None: nn.init.zeros_(m.bias)

class TrendCNN(nn.Module):
    def __init__(self, n_days=20):
        super().__init__()
        if n_days == 5:    blocks, vs, vd = 2, 1, 1
        elif n_days == 20: blocks, vs, vd = 3, 3, 2
        else:              blocks, vs, vd = 4, 3, 3
        layers, in_c, out_c = [], 1, 64
        for i in range(blocks):
            stride   = (vs, 1) if i == 0 else (1, 1)
            dilation = (vd, 1) if i == 0 else (1, 1)
            layers += [nn.Conv2d(in_c, out_c, (5,3), stride=stride, padding=(2,1), dilation=dilation),
                       nn.BatchNorm2d(out_c), nn.LeakyReLU(0.01), nn.MaxPool2d((2,1))]
            in_c, out_c = out_c, out_c*2
        self.features = nn.Sequential(*layers)
        H, W = img_shape(n_days)
        flat = self.features(torch.zeros(1,1,H,W)).view(1,-1).size(1)
        self.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(flat, 2))
        self.apply(xavier_init)
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x.view(x.size(0), -1))

## 6. Training engine — per-member checkpointing + explicit teardown

Every member writes `preds/pred_I{n}_R{h}_s{seed}.npy` on completion, so a member trained on the Mac and a member trained on Kaggle combine freely; re-running skips whatever already exists.

**Memory hygiene** (this is what stops the I60 creep): after each member the model, optimizer, all three DataLoaders and their persistent worker processes are dropped, the memmap handle is released, `gc.collect()` runs, and the device cache is emptied. RSS and GPU memory print every epoch — if they still climb monotonically, you'll see exactly where.

In [35]:
def _mk_loader(ds, shuffle, device):
    kw = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
              pin_memory=(device.startswith('cuda')))
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(ds, **kw)

def _shutdown(loader):
    """Persistent workers survive the loader unless the iterator is dropped first."""
    try:
        it = getattr(loader, '_iterator', None)
        if it is not None:
            it._shutdown_workers(); loader._iterator = None
    except Exception:
        pass

def pred_path(n_days, horizon, seed):
    return os.path.join(PRED_DIR, f'pred_I{n_days}_R{horizon}_s{seed}.npy')

def train_and_predict(job):
    n_days, horizon, dat, shape, tr, trl, va, val, te, device, seed, tag = job
    out = pred_path(n_days, horizon, seed)
    if os.path.exists(out):                                  # RESUME: this member is already done
        p = np.load(out)
        log(f"{tag} cached ({len(p):,} preds) -> skipping", tag='·')
        return p

    torch.manual_seed(seed); np.random.seed(seed)
    dev = torch.device(device)
    if dev.type == 'cuda': torch.cuda.set_device(dev)

    ds_tr = ImgDataset(dat, shape, tr, trl)
    ds_va = ImgDataset(dat, shape, va, val)
    ds_te = ImgDataset(dat, shape, te)
    l_tr = _mk_loader(ds_tr, True,  device)
    l_va = _mk_loader(ds_va, False, device)
    l_te = _mk_loader(ds_te, False, device)

    model     = TrendCNN(n_days).to(dev)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    amp    = USE_AMP and dev.type == 'cuda'
    scaler = torch.cuda.amp.GradScaler(enabled=amp)

    log(f"{tag} start on {device} | train {len(tr):,} val {len(va):,} test {len(te):,} | {mem_str(dev)}")
    best, patience_ctr, t_start = float('inf'), 0, time.time()
    ckpt = tempfile.NamedTemporaryFile(suffix='.pt', delete=False).name   # unique per member -> no races

    for epoch in range(EPOCHS):
        t0 = time.time(); model.train(); tr_loss = 0.0
        for imgs, y in l_tr:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)    # uint8 -> float ON device
            y    = y.to(dev, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            if amp:
                with torch.autocast('cuda'):
                    loss = criterion(model(imgs), y)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            else:
                loss = criterion(model(imgs), y)
                loss.backward(); optimizer.step()
            tr_loss += loss.item()
        tr_loss /= max(1, len(l_tr))

        model.eval(); va_loss = 0.0
        with torch.no_grad():
            for imgs, y in l_va:
                imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
                y    = y.to(dev, non_blocking=True)
                if amp:
                    with torch.autocast('cuda'):
                        va_loss += criterion(model(imgs), y).item()
                else:
                    va_loss += criterion(model(imgs), y).item()
        va_loss /= max(1, len(l_va))

        mark = ''
        if va_loss < best:
            best, patience_ctr = va_loss, 0
            torch.save(model.state_dict(), ckpt); mark = 'best'
        else:
            patience_ctr += 1
            mark = f'patience {patience_ctr}/{PATIENCE}'
        log(f"{tag} ep{epoch+1:03d} train {tr_loss:.4f} val {va_loss:.4f} "
            f"{time.time()-t0:5.1f}s {mark} | {mem_str(dev)}")
        if patience_ctr >= PATIENCE:
            log(f"{tag} early stop at epoch {epoch+1} (best val {best:.4f})")
            break

    model.load_state_dict(torch.load(ckpt, map_location=dev, weights_only=True))
    os.remove(ckpt)

    model.eval(); probs = []
    with torch.no_grad():
        for imgs in l_te:
            imgs = imgs.to(dev, non_blocking=True).float().div_(255.0)
            if amp:
                with torch.autocast('cuda'):
                    p = torch.softmax(model(imgs), 1)[:, 1]
            else:
                p = torch.softmax(model(imgs), 1)[:, 1]
            probs.append(p.float().cpu().numpy())
    probs = np.concatenate(probs) if probs else np.array([])
    np.save(out, probs)                                        # checkpoint: survives a crash / machine switch

    # ---- teardown: the anti-leak block ----
    for l in (l_tr, l_va, l_te): _shutdown(l)
    for d in (ds_tr, ds_va, ds_te): d.close()
    del l_tr, l_va, l_te, ds_tr, ds_va, ds_te, model, optimizer, criterion, scaler
    free_memory(dev)

    log(f"{tag} done in {(time.time()-t_start)/60:.1f} min | {len(probs):,} preds -> {os.path.basename(out)} "
        f"| {mem_str(dev)}")
    return probs

## 7. Portfolio evaluation — JKX Table 3, plus the real Han–Zhou–Zhu TREND

**TREND is computed here, not in data prep**, because the genuine Han–Zhou–Zhu (2016) trend factor is a *fitted expected-return forecast*, not the moving-average crossover vote the old code used. At each rebalance date `t`:

1. cross-sectionally regress the `H`-day forward return on the normalized moving averages `MA{3,5,10,20,50,100,200}` → `beta_t`;
2. forecast `TREND_i,t = mean(previous n_beta betas) · MA_i,t`.

`beta_t` needs a realized return, so it only becomes *knowable* at `t+H`. Rebalance dates are spaced exactly `H` apart, so any beta from `s < t` is realized by `t` — the forecast uses **only already-realized information**. Verified: corrupting the last date's returns leaves every earlier forecast bit-identical.

The portfolio itself: rank names into 10 deciles each rebalance date, hold the equal-weight long-short (D10 − D1) for `H` days, **rebalancing every `H` trading days so periods never overlap**. Equal-weighted only — the paper's value-weighted panel needs market cap, which this OHLCV parquet doesn't carry.

In [36]:
def add_trend_factor(ev, horizon, n_beta=12, min_names=30):
    """Han-Zhou-Zhu (2016) trend factor. Real-time: no look-ahead."""
    ev = ev.copy(); ev['TREND'] = np.nan
    dates = np.sort(ev['Date'].unique())
    rebal = list(dates[::horizon])

    betas = {}
    for t in rebal:                                        # beta_t needs r(t -> t+H): only KNOWN at t+H
        g = ev[ev['Date'] == t].dropna(subset=MA_COLS + ['Fwd_Ret'])
        if len(g) < min_names: continue
        X = np.column_stack([np.ones(len(g)), g[MA_COLS].values])
        b, *_ = np.linalg.lstsq(X, g['Fwd_Ret'].values, rcond=None)
        betas[t] = b

    known = []                                             # betas already realized before the current date
    for t in rebal:
        if len(known) >= n_beta:                           # burn-in: first n_beta dates get no forecast
            bbar = np.mean(known[-n_beta:], axis=0)
            g  = ev.loc[ev['Date'] == t, MA_COLS]
            ok = g.notna().all(axis=1)
            if ok.any():
                X = np.column_stack([np.ones(int(ok.sum())), g[ok].values])
                ev.loc[g[ok].index, 'TREND'] = X @ bbar
        if t in betas:
            known.append(betas[t])                         # realized at t+H -> usable from the next date on
    return ev

def portfolio_performance(signal, df, horizon, min_names=20):
    # The rebalance calendar is a property of the STRATEGY, so derive it from the FULL date grid.
    # (Deriving it after dropna would re-thin an already-spaced grid whenever a signal has a
    #  warm-up gap -- e.g. TREND's burn-in -- collapsing the sample to a couple of periods.)
    dates = np.sort(df['Date'].unique())
    rebal = set(dates[::horizon].tolist())                    # non-overlapping holding periods
    d = df[df['Date'].isin(rebal)].dropna(subset=[signal, 'Fwd_Ret'])   # signals NaN in warm-up -> skip date
    if d.empty:
        return dict(AvgHoldRet=np.nan, tstat=np.nan, AnnReturn=np.nan, AnnSharpe=np.nan, NPeriods=0)
    rets = []
    for _, g in d.groupby('Date'):
        if len(g) < min_names:                                # too thin to define 10 deciles
            continue
        dec = pd.qcut(g[signal].rank(method='first'), 10, labels=False)
        rets.append(g.loc[dec == 9, 'Fwd_Ret'].mean() - g.loc[dec == 0, 'Fwd_Ret'].mean())
    r = np.asarray(rets, float); r = r[~np.isnan(r)]; n = len(r)
    if n < 2:
        return dict(AvgHoldRet=(r.mean() if n else np.nan), tstat=np.nan,
                    AnnReturn=np.nan, AnnSharpe=np.nan, NPeriods=n)
    ppy = 252.0 / horizon
    mu, sd = r.mean(), r.std(ddof=1)
    return dict(AvgHoldRet=mu,
                tstat=(mu/(sd/np.sqrt(n)) if sd > 0 else np.nan),
                AnnReturn=mu*ppy,
                AnnSharpe=(mu/sd*np.sqrt(ppy) if sd > 0 else np.nan),
                NPeriods=n)

## 8. Run

Resumes automatically: configs already in `results.csv` are skipped, and within a config any member with a cached `.npy` is skipped. So you can stop this at any point, move machines, and re-run.

In [37]:
# ---- generate any Omissing stores (no-op on Kaggle if the dataset is mounted) ----
for n_days in IMAGE_SIZES:
    dat, meta = store_paths(n_days)
    if os.path.exists(dat) and os.path.exists(meta):
        continue
    if ON_KAGGLE:
        log(f"N={n_days}: MISSING from IMG_DIR and generation on Kaggle is slow -- "
            f"generate on the Mac and upload. Skipping this size.")
        continue
    build_store(df_market, n_days)


In [24]:
IMAGE_SIZES = [20]
HORIZONS    = [5,20,60]
# ---- resume: what's already finished? ----
done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(prev['Config'].unique())
    log(f"resuming: {len(done)} configs already in {RESULTS_CSV} -> {sorted(done)}")
    results = prev.to_dict('records')
else:
    results = []

for n_days in IMAGE_SIZES:
    store = load_store(n_days)
    if store is None:
        log(f"N={n_days}: no store found, skipping."); continue

    for horizon in HORIZONS:
        cfg = f"I{n_days}/R{horizon}"
        if cfg in done:
            log(f"{cfg}: already complete, skipping."); continue
        log("=" * 66)
        log(f"CONFIG {cfg}  (ensemble {ENSEMBLE_SIZE}, {MAX_PARALLEL} concurrent on {BACKEND})")
        log("=" * 66)

        rows, label, base = label_for_horizon(store, df_market, horizon)
        te     = test_positions(base)
        tv_all = trainval_positions(base)
        if len(tv_all) == 0 or len(te) == 0:
            log(f"{cfg}: empty train or test split, skipping."); continue
        te_rows   = rows[te]
        eval_base = base.iloc[te].reset_index(drop=True)     # row-for-row aligned with every member's preds
        log(f"{cfg}: train/val pool {len(tv_all):,} | test {len(te):,} "
            f"| test dates {eval_base['Date'].min()} -> {eval_base['Date'].max()}")

        jobs = []
        for i in range(ENSEMBLE_SIZE):
            seed = 1000 + i
            tr, va = subsample_split(tv_all, seed if BAGGING else SHARED_SPLIT_SEED)
            jobs.append((n_days, horizon, store['dat'], store['shape'],
                         rows[tr], label[tr], rows[va], label[va], te_rows,
                         DEVICES[i % len(DEVICES)], seed, f"[{cfg} m{i+1}/{ENSEMBLE_SIZE}]"))

        preds = [None] * ENSEMBLE_SIZE
        t_cfg = time.time()
        if MAX_PARALLEL > 1:
            with cf.ThreadPoolExecutor(max_workers=MAX_PARALLEL) as ex:
                fut = {ex.submit(train_and_predict, j): k for k, j in enumerate(jobs)}
                for f in cf.as_completed(fut):
                    preds[fut[f]] = f.result()               # raises here if a member failed -- don't hide it
        else:
            for k, j in enumerate(jobs):
                preds[k] = train_and_predict(j)

        lens = {len(p) for p in preds}
        if lens != {len(te)}:
            raise RuntimeError(f"{cfg}: prediction lengths {lens} != test rows {len(te)}. "
                               f"Stale .npy from a different run? Delete {PRED_DIR} for this config.")

        eval_base['CNN_Prob'] = np.mean(preds, axis=0)
        eval_base = add_trend_factor(eval_base, horizon)      # real HZZ trend factor for THIS horizon
        log(f"{cfg}: ensemble complete in {(time.time()-t_cfg)/60:.1f} min | scoring...")

        for strat in ['CNN_Prob','MOM','STR','WSTR','TREND']:
            perf = portfolio_performance(strat, eval_base, horizon)
            results.append({'Config': cfg, 'Strategy': strat, **perf})
            log(f"  {strat:9s} AnnRet {perf['AnnReturn']:+.4f}  Sharpe {perf['AnnSharpe']:+.3f}  "
                f"t {perf['tstat']:+.2f}  periods {perf['NPeriods']}")

        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)    # checkpoint after every config
        del preds, eval_base, rows, label, base, te, tv_all, te_rows
        free_memory()
        log(f"{cfg}: saved -> {RESULTS_CSV} | mem={mem_str()}")

    del store
    free_memory()

df_results = pd.DataFrame(results)
log(f"ALL DONE in {(time.time()-_T0)/60:.1f} min | mem={mem_str()}")
display(df_results)

[00:00:03.4] resuming: 3 configs already in ./results.csv -> ['I5/R20', 'I5/R5', 'I5/R60']
[00:00:03.6] N=20: 2,583,356 images on disk (9.9 GB), 2,583,356 usable | mem=rss 2.5G mps 0.0G
[00:00:03.6] ==================================================================
[00:00:03.6] CONFIG I20/R5  (ensemble 5, 1 concurrent on mps)
[00:00:03.6] ==================================================================
[00:00:04.2] I20/R5: train/val pool 480,595 | test 2,100,235 | test dates 2005-01-03 -> 2026-06-23
[00:00:04.4] [I20/R5 m1/5] start on mps | train 336,416 val 144,179 test 2,100,235 | rss 2.8G mps 0.0G
[00:04:48.0] [I20/R5 m1/5] ep001 train 0.9681 val 0.7158 283.5s best | rss 0.1G mps 0.1G
[00:09:27.8] [I20/R5 m1/5] ep002 train 0.8050 val 0.7033 279.8s best | rss 0.1G mps 0.1G
[00:14:11.1] [I20/R5 m1/5] ep003 train 0.7344 val 0.6966 283.3s best | rss 0.1G mps 0.1G
[00:18:56.0] [I20/R5 m1/5] ep004 train 0.7067 val 0.6935 284.9s best | rss 0.1G mps 0.1G
[00:23:42.2] [I20/R5 m1/5] ep005 t

,Config,Strategy,AvgHoldRet,tstat,AnnReturn,AnnSharpe,NPeriods
0,I5/R5,CNN_Prob,0.001220,2.338360,0.061503,0.504910,1081
1,I5/R5,MOM,-0.000676,-0.596154,-0.034058,-0.128724,1081
2,I5/R5,STR,0.001439,1.405383,0.072508,0.303457,1081
3,I5/R5,WSTR,0.002008,2.161891,0.101216,0.466806,1081
4,I5/R5,TREND,-0.000284,-0.306242,-0.014331,-0.066495,1069
5,I5/R20,CNN_Prob,0.000132,0.074470,0.001665,0.016087,270
6,I5/R20,MOM,-0.002517,-0.545726,-0.031720,-0.117890,270
7,I5/R20,STR,0.002736,0.722571,0.034477,0.156093,270
8,I5/R20,WSTR,0.003341,0.922058,0.042102,0.199187,270
9,I5/R20,TREND,0.004510,1.119833,0.056829,0.247473,258


## 9. Results

In [38]:
df_results = pd.read_csv(RESULTS_CSV)
piv = df_results.pivot(index='Config', columns='Strategy', values='AnnSharpe')
order = [f'I{i}/R{h}' for i in [5,20,60] for h in [5,20,60] if f'I{i}/R{h}' in piv.index]
print("Annualized Sharpe (equal-weight decile long-short, non-overlapping):")
display(piv.loc[order][['CNN_Prob','MOM','STR','WSTR','TREND']].round(3))
print("\nCNN detail:")
display(df_results[df_results.Strategy=='CNN_Prob']
        .set_index('Config')[['AnnReturn','AnnSharpe','tstat','NPeriods']].round(3))

Annualized Sharpe (equal-weight decile long-short, non-overlapping):


Strategy,CNN_Prob,MOM,STR,WSTR,TREND
Config,,,,,
I5/R5,0.505,-0.129,0.303,0.467,-0.066
I5/R20,0.016,-0.118,0.156,0.199,0.247
I5/R60,-0.359,-0.103,0.189,-0.055,-0.381
I20/R5,-0.084,-0.129,0.303,0.469,-0.066
I20/R20,0.322,-0.118,0.156,0.195,0.247
I20/R60,0.084,-0.103,0.189,-0.056,-0.381



CNN detail:


,AnnReturn,AnnSharpe,tstat,NPeriods
Config,,,,
I5/R5,0.062,0.505,2.338,1081
I5/R20,0.002,0.016,0.074,270
I5/R60,-0.027,-0.359,-1.663,90
I20/R5,-0.009,-0.084,-0.391,1081
I20/R20,0.031,0.322,1.492,270
I20/R60,0.006,0.084,0.388,90


In [39]:
def subperiod_report(splits=None):
    if splits is None:
        splits = {
            'full  2005-2026'          : ('2005-01-01', '2026-12-31'),
            'early 2005-2014 (low cov)': ('2005-01-01', '2014-12-31'),
            'late  2015-2026 (high cov)': ('2015-01-01', '2026-12-31'),
        }
    out = []
    for n_days in IMAGE_SIZES:
        store = load_store(n_days)
        if store is None:
            continue
        for horizon in HORIZONS:
            cfg = f"I{n_days}/R{horizon}"
            # rebuild eval_base exactly as the main loop did
            rows, label, base = label_for_horizon(store, df_market, horizon)
            te = test_positions(base)
            if len(te) == 0:
                continue
            eval_base = base.iloc[te].reset_index(drop=True)

            # pull the CACHED member predictions -- no training
            preds = []
            for i in range(ENSEMBLE_SIZE):
                p = pred_path(n_days, horizon, 1000 + i)
                if os.path.exists(p):
                    preds.append(np.load(p))
            if not preds or {len(p) for p in preds} != {len(te)}:
                log(f"{cfg}: cached predictions missing/stale, skipping.")
                continue
            eval_base['CNN_Prob'] = np.mean(preds, axis=0)
            eval_base = add_trend_factor(eval_base, horizon)

            for name, (a, b) in splits.items():
                sub = eval_base[(eval_base['Date'] >= a) & (eval_base['Date'] <= b)]
                if sub.empty:
                    continue
                for s in ['CNN_Prob', 'MOM', 'STR', 'WSTR', 'TREND']:
                    perf = portfolio_performance(s, sub, horizon)
                    out.append({'Config': cfg, 'Period': name, 'Strategy': s, **perf})
        del store
        free_memory()

    df = pd.DataFrame(out)
    df.to_csv(os.path.join(OUT_DIR, 'subperiod_results.csv'), index=False)

    print("\n" + "=" * 74)
    print("SUB-PERIOD ROBUSTNESS — does the CNN edge survive where coverage is HIGH?")
    print("=" * 74)
    cnn = df[df.Strategy == 'CNN_Prob'].pivot(index='Config', columns='Period',
                                              values='AnnSharpe')
    display(cnn.round(3))
    print("\nHOW TO READ THIS")
    print("  edge in BOTH halves  -> survivorship is not driving it. Say so, with this table.")
    print("  edge ONLY in 'early' -> it is likely an artifact of the survivor-selected universe.")
    print("  edge ONLY in 'late'  -> not survivorship; more likely a regime/period effect.")
    return df

sub = subperiod_report()


[00:00:13.7] N=5: 2,586,955 images on disk (1.2 GB), 2,586,955 usable | mem=rss 2.1G mps 0.0G
[00:00:36.3] N=20: 2,583,356 images on disk (9.9 GB), 2,583,356 usable | mem=rss 3.9G mps 0.0G
[00:00:58.5] N=60: 2,573,322 images on disk (44.5 GB), 2,573,322 usable | mem=rss 3.7G mps 0.0G
[00:00:59.1] I60/R5: cached predictions missing/stale, skipping.
[00:00:59.5] I60/R20: cached predictions missing/stale, skipping.
[00:00:59.9] I60/R60: cached predictions missing/stale, skipping.

SUB-PERIOD ROBUSTNESS — does the CNN edge survive where coverage is HIGH?


Period,early 2005-2014 (low cov),full 2005-2026,late 2015-2026 (high cov)
Config,,,
I20/R20,0.340,0.322,0.063
I20/R5,0.065,-0.084,0.212
I20/R60,-0.011,0.084,0.038
I5/R20,0.025,0.016,0.188
I5/R5,0.650,0.505,0.378
I5/R60,-0.537,-0.359,0.026



HOW TO READ THIS
  edge in BOTH halves  -> survivorship is not driving it. Say so, with this table.
  edge ONLY in 'early' -> it is likely an artifact of the survivor-selected universe.
  edge ONLY in 'late'  -> not survivorship; more likely a regime/period effect.


In [40]:
# ---- Coverage-matched test: is it the COUNT of names, or the SELECTION of names? ----
# Rebuilds eval_base from cached predictions (no retraining), then randomly thins the
# high-coverage late years down to the early years' coverage and re-scores.

def rebuild_eval_base(n_days, horizon):
    store = load_store(n_days)
    if store is None:
        return None
    rows, label, base = label_for_horizon(store, df_market, horizon)
    te = test_positions(base)
    if len(te) == 0:
        return None
    eb = base.iloc[te].reset_index(drop=True)
    preds = [np.load(pred_path(n_days, horizon, 1000 + i))
             for i in range(ENSEMBLE_SIZE) if os.path.exists(pred_path(n_days, horizon, 1000 + i))]
    if not preds or {len(p) for p in preds} != {len(te)}:
        log(f"I{n_days}/R{horizon}: cached preds missing/stale")
        return None
    eb['CNN_Prob'] = np.mean(preds, axis=0)
    eb = add_trend_factor(eb, horizon)
    del store
    free_memory()
    return eb


def coverage_matched_test(n_days, horizon, target=0.64, n_trials=20, seed=0):
    eb = rebuild_eval_base(n_days, horizon)
    if eb is None:
        return
    rng  = np.random.default_rng(seed)
    late = eb[eb['Date'] >= '2015-01-01']
    full = portfolio_performance('CNN_Prob', late, horizon)['AnnSharpe']

    shs = []
    for _ in range(n_trials):
        keep = []
        for _, g in late.groupby('Date'):                  # thin each date to `target` of its names
            idx = g.index.values
            keep.append(rng.choice(idx, size=max(20, int(target * len(idx))), replace=False))
        sub = late.loc[np.concatenate(keep)]
        shs.append(portfolio_performance('CNN_Prob', sub, horizon)['AnnSharpe'])

    print(f"I{n_days}/R{horizon}  late period (2015-2026)")
    print(f"  full universe            : Sharpe {full:.3f}")
    print(f"  randomly thinned to {target:.0%} : Sharpe {np.mean(shs):.3f} ± {np.std(shs):.3f}  (n={n_trials})")
    print(f"  gap                      : {np.mean(shs) - full:+.3f}")


# run it on your headline config (the one where the CNN actually works)
coverage_matched_test(5, 5)

[00:01:00.1] N=5: 2,586,955 images on disk (1.2 GB), 2,586,955 usable | mem=rss 3.7G mps 0.0G
I5/R5  late period (2015-2026)
  full universe            : Sharpe 0.378
  randomly thinned to 64% : Sharpe 0.334 ± 0.059  (n=20)
  gap                      : -0.044


In [42]:
# =============================================================================
# SCIENTIFIC ANALYSIS  —  Stage A (characterize) + Stage B (corrected grid) in one cell.
# Paste at the END of notebook_final_unified.ipynb. Run §1-§7 first (definitions).
# Uses CACHED predictions -> no retraining if pred_*.npy exist for the configs you ran.
#
# DESIGN (why it's built this way):
#   * The 9-config grid IS the search you performed. The scientific question is NOT
#     "is I5/R5 good?" (it was SELECTED for being the max t-stat -> circular). It is
#     "does ANY config survive multiple-testing correction across the family searched?"
#   * I5/R5 is then CHARACTERIZED descriptively (costs, stability, monotonicity,
#     block-bootstrap CI), never re-tested for significance on the same data.
#   * The survivorship early/late split is reported as the headline finding.
# =============================================================================
from math import erf, sqrt

def align_store(m, df, verbose=False):
    """Boolean mask of store rows whose image still matches the current parquet."""
    pos_df = (df.sort_values(['Ticker', 'Date'])
                .assign(pos=lambda x: x.groupby('Ticker').cumcount(),
                        end_date=lambda x: x['Date'].dt.strftime('%Y-%m-%d'))
                [['Ticker', 'end_date', 'pos']]
                .rename(columns={'Ticker': 'ticker'}))
    mm = m.merge(pos_df, on=['ticker', 'end_date'], how='left')
    ok = mm['pos'].notna() & (mm['pos'].values == mm['end_idx'].values)
    return ok.values, mm
    
# ---------- shared machinery ----------
def _ls_series(df, sig, H, min_names=20, cost_bps=0.0):
    d = df.dropna(subset=[sig, 'Fwd_Ret']); dts = np.sort(d['Date'].unique())
    d = d[d['Date'].isin(set(dts[::H].tolist()))]; out = {}
    for dt, g in d.groupby('Date'):
        if len(g) < min_names: continue
        dec = pd.qcut(g[sig].rank(method='first'), 10, labels=False)
        out[dt] = g.loc[dec==9,'Fwd_Ret'].mean() - g.loc[dec==0,'Fwd_Ret'].mean() - 2*cost_bps/1e4
    return pd.Series(out).sort_index()

def _sharpe(r, H):
    r = r.dropna()
    return np.nan if len(r) < 2 or r.std(ddof=1)==0 else r.mean()/r.std(ddof=1)*np.sqrt(252.0/H)

def _tstat(r):
    r = r.dropna()
    return np.nan if len(r) < 2 or r.std(ddof=1)==0 else r.mean()/(r.std(ddof=1)/np.sqrt(len(r)))

def _p_two_sided(t):
    return np.nan if np.isnan(t) else 2*(1-0.5*(1+erf(abs(t)/sqrt(2))))

def _block_boot_ci(r, H, n_boot=2000, seed=0):
    r = r.dropna().values; n = len(r)
    if n < 10: return (np.nan, np.nan)
    avg_block = max(2, int(n**(1/3))); p = 1.0/avg_block
    rng = np.random.default_rng(seed); ann = np.sqrt(252.0/H); est = np.empty(n_boot)
    for b in range(n_boot):
        idx = np.empty(n, dtype=int); i = 0
        while i < n:
            start = rng.integers(0, n); L = rng.geometric(p)
            for k in range(L):
                if i >= n: break
                idx[i] = (start+k) % n; i += 1
        s = r[idx]; est[b] = s.mean()/s.std(ddof=1)*ann if s.std(ddof=1)>0 else 0.0
    return (np.percentile(est, 2.5), np.percentile(est, 97.5))

def _bh_reject(pvals, alpha=0.05):
    p = np.array([x if not np.isnan(x) else 1.0 for x in pvals]); m = len(p)
    order = np.argsort(p); passed = p[order] <= (np.arange(1, m+1)/m)*alpha
    rej = np.zeros(m, bool)
    if passed.any(): rej[order[:np.where(passed)[0].max()+1]] = True
    return rej

def _rebuild(n_days, horizon):
    store = load_store(n_days, df_market)
    rows, label, base = label_for_horizon(store, df_market, horizon)
    te = test_positions(base); eb = base.iloc[te].reset_index(drop=True)
    ps = [np.load(pred_path(n_days, horizon, 1000+i)) for i in range(ENSEMBLE_SIZE)
          if os.path.exists(pred_path(n_days, horizon, 1000+i))]
    del store; free_memory()
    if not ps or {len(p) for p in ps} != {len(te)}: return None
    eb['CNN_Prob'] = np.mean(ps, axis=0)
    return add_trend_factor(eb, horizon)

# which configs actually have cached predictions?
CONFIGS = [(n, h) for n in IMAGE_SIZES for h in HORIZONS
           if os.path.exists(pred_path(n, h, 1000))]
log(f"configs with cached predictions: {[f'I{n}/R{h}' for n,h in CONFIGS]}")

# ============================================================================
print("\n" + "="*72)
print("STAGE B  —  FULL-GRID MULTIPLE-TESTING CORRECTION  (the primary result)")
print("="*72)
print("The family of tests IS the 9-config search. Report the CORRECTED verdict.\n")

grid = []
for n, h in CONFIGS:
    eb = _rebuild(n, h)
    if eb is None: continue
    r = _ls_series(eb, 'CNN_Prob', h); t = _tstat(r)
    grid.append({'Config': f'I{n}/R{h}', 'Sharpe': _sharpe(r, h), 'tstat': t,
                 'raw_p': _p_two_sided(t), 'N': len(r.dropna())})
gdf = pd.DataFrame(grid)
gdf['bonferroni_p'] = np.minimum(gdf['raw_p'] * len(gdf), 1.0)
gdf['BH_reject']    = _bh_reject(gdf['raw_p'].values)
gdf = gdf.sort_values('raw_p').reset_index(drop=True)
print(gdf.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
n_survive = int(gdf['BH_reject'].sum())
print(f"\n  tests in family        : {len(gdf)}")
print(f"  survive BH (FDR 5%)    : {n_survive}")
print(f"  survive Bonferroni     : {int((gdf['bonferroni_p']<0.05).sum())}")
print(f"\n  PRIMARY VERDICT: " + (
      f"{n_survive} config(s) survive correction -- a genuine, if modest, effect."
      if n_survive else
      "NO config survives correction across the searched family. The apparent\n"
      "  per-config signals are consistent with multiple-testing noise. This is the\n"
      "  honest headline, and it is a real finding, not a failure."))

# ============================================================================
print("\n" + "="*72)
print("STAGE A  —  CHARACTERIZE THE STRONGEST CONFIG  (descriptive, NOT a p-value)")
print("="*72)
best = gdf.iloc[0]['Config']; bn, bh_ = int(best[1:best.index('/')]), int(best[best.index('R')+1:])
print(f"strongest raw config: {best}  (selected post hoc -> characterize, don't re-test)\n")
eb = _rebuild(bn, bh_)

if eb is not None:
    r = _ls_series(eb, 'CNN_Prob', bh_)
    lo, hi = _block_boot_ci(r, bh_)
    print(f"[A0] Block-bootstrap Sharpe CI (honours autocorrelation, the t-stat does not)")
    print(f"     Sharpe {_sharpe(r,bh_):+.3f}   95% CI [{lo:+.3f}, {hi:+.3f}]   "
          f"-> {'excludes 0' if (lo>0 or hi<0) else 'STRADDLES 0 (weak)'}\n")

    print("[A1] Transaction costs")
    for bps in [0,5,10,20]:
        rc = _ls_series(eb,'CNN_Prob',bh_,cost_bps=bps)
        print(f"     {bps:2d} bps/side : Sharpe {_sharpe(rc,bh_):+.3f}")
    print()

    print("[A2] Rolling 3-year Sharpe")
    idx = pd.to_datetime(r.index); shs=[]
    for y0 in sorted(set(idx.year)):
        sub = r[(idx.year>=y0)&(idx.year<y0+3)]
        if len(sub)>20: shs.append(_sharpe(sub,bh_))
    print(f"     windows: {len(shs)} | positive: {np.mean([s>0 for s in shs]):.0%} | "
          f"min {min(shs):+.2f} max {max(shs):+.2f}\n")

    print("[A3] Decile monotonicity")
    d = eb.dropna(subset=['CNN_Prob','Fwd_Ret']); dts=np.sort(d['Date'].unique())
    d = d[d['Date'].isin(set(dts[::bh_].tolist()))]; means=np.zeros(10); c=0
    for _,g in d.groupby('Date'):
        if len(g)<20: continue
        dec=pd.qcut(g['CNN_Prob'].rank(method='first'),10,labels=False)
        for k in range(10): means[k]+=g.loc[dec==k,'Fwd_Ret'].mean()
        c+=1
    means/=c; rho=np.corrcoef(np.arange(10),means)[0,1]
    print(f"     decile means (bps): " + " ".join(f"{m*1e4:+.0f}" for m in means))
    print(f"     rank corr decile->return: {rho:+.2f}  (>0.5 = coherent; flat = tail-only)\n")

    print("[A4] Long vs short leg")
    longs,shorts=[],[]
    for _,g in d.groupby('Date'):
        if len(g)<20: continue
        dec=pd.qcut(g['CNN_Prob'].rank(method='first'),10,labels=False); mk=g['Fwd_Ret'].mean()
        longs.append(g.loc[dec==9,'Fwd_Ret'].mean()-mk); shorts.append(mk-g.loc[dec==0,'Fwd_Ret'].mean())
    print(f"     long  (D9-mkt): Sharpe {_sharpe(pd.Series(longs),bh_):+.3f}")
    print(f"     short (mkt-D0): Sharpe {_sharpe(pd.Series(shorts),bh_):+.3f}")

# ============================================================================
print("\n" + "="*72)
print("HEADLINE  —  SURVIVORSHIP DECOMPOSITION  (the interesting scientific result)")
print("="*72)
splits = {'early 2005-2014 (low coverage)': ('2005-01-01','2014-12-31'),
          'late  2015-2026 (high coverage)': ('2015-01-01','2026-12-31')}
for n, h in CONFIGS:
    eb = _rebuild(n, h)
    if eb is None: continue
    parts = {k: _sharpe(_ls_series(eb[(eb.Date>=a)&(eb.Date<=b)],'CNN_Prob',h), h)
             for k,(a,b) in splits.items()}
    full = _sharpe(_ls_series(eb,'CNN_Prob',h), h)
    print(f"  I{n}/R{h}: full {full:+.2f} | early {parts['early 2005-2014 (low coverage)']:+.2f} "
          f"| late {parts['late  2015-2026 (high coverage)']:+.2f}")
print("\n  edge concentrated in EARLY (low-coverage) years and absent/negative in the")
print("  clean LATE years  ->  the apparent effect is largely a survivorship artifact.")

pd.DataFrame(grid).to_csv(os.path.join(OUT_DIR,'corrected_grid.csv'), index=False)
print(f"\nsaved -> {os.path.join(OUT_DIR,'corrected_grid.csv')}")


[00:02:48.6] configs with cached predictions: ['I5/R5', 'I5/R20', 'I5/R60', 'I20/R5', 'I20/R20', 'I20/R60']

STAGE B  —  FULL-GRID MULTIPLE-TESTING CORRECTION  (the primary result)
The family of tests IS the 9-config search. Report the CORRECTED verdict.

[00:02:49.5] N=5: 2,586,955 images on disk (1.2 GB), 2,586,955 usable | mem=rss 2.7G mps 0.0G
[00:03:02.4] N=5: 2,586,955 images on disk (1.2 GB), 2,586,955 usable | mem=rss 4.0G mps 0.0G
[00:03:06.6] N=5: 2,586,955 images on disk (1.2 GB), 2,586,955 usable | mem=rss 3.6G mps 0.0G
[00:03:09.0] N=20: 2,583,356 images on disk (9.9 GB), 2,583,356 usable | mem=rss 4.1G mps 0.0G
[00:03:21.6] N=20: 2,583,356 images on disk (9.9 GB), 2,583,356 usable | mem=rss 4.6G mps 0.0G
[00:03:25.8] N=20: 2,583,356 images on disk (9.9 GB), 2,583,356 usable | mem=rss 4.7G mps 0.0G
 Config  Sharpe   tstat  raw_p    N  bonferroni_p  BH_reject
  I5/R5  0.5049  2.3384 0.0194 1081        0.1162      False
 I5/R60 -0.3593 -1.6630 0.0963   90        0.5778      

In [43]:
# =============================================================================
# SIZE-TERCILE SPLIT  —  pre-registered within-universe test.
# Paste at the END of notebook_final_unified.ipynb (after §7 definitions + the
# scientific-analysis cell). Uses CACHED predictions -> no retraining.
#
# PRE-REGISTERED (fixed before looking at output):
#   HYPOTHESIS : within the S&P 500, the CNN long-short Sharpe rises as market
#                cap falls (big -> mid -> small tercile).
#   CONFIGS    : I5/R5 and I20/R20 only.
#   SIZE PROXY : trailing-63d median dollar volume (Close*Volume), SHIFTED one day
#                so it is known strictly before the rebalance (no look-ahead).
#   TERCILES   : re-formed cross-sectionally each rebalance date; decile long-short
#                is formed WITHIN each tercile.
#   REPORT     : big/mid/small Sharpe per config, whatever it shows.
#
# CAVEAT to state in the writeup: this is the smaller third of LARGE caps, NOT true
# small caps. It tests the size gradient within the universe you have; it cannot
# reach the micro-cap region where JKX's effect is strongest (that needs CRSP/WRDS).
# =============================================================================

PREREG_CONFIGS = [(5, 5), (20, 20)]      # locked

def size_tercile_ls(eval_base, price_df, signal, H, min_per_tercile=15, cost_bps=0.0):
    pv = price_df.copy()
    pv['DV'] = pv['Close'] * pv['Volume']
    pv = pv.sort_values(['Ticker', 'Date'])
    pv['SIZE'] = (pv.groupby('Ticker')['DV']
                    .transform(lambda s: s.rolling(63, min_periods=20).median().shift(1)))  # strictly past
    size_lookup = pv.set_index(['Ticker', 'Date'])['SIZE']

    d = eval_base.dropna(subset=[signal, 'Fwd_Ret']).copy()
    d['Date'] = pd.to_datetime(d['Date'])
    d['SIZE'] = size_lookup.reindex(pd.MultiIndex.from_arrays([d['Ticker'], d['Date']])).values
    d = d.dropna(subset=['SIZE'])

    dates = np.sort(d['Date'].unique())
    d = d[d['Date'].isin(set(pd.Series(dates[::H]).tolist()))]

    ser = {'big': {}, 'mid': {}, 'small': {}}
    for dt, g in d.groupby('Date'):
        if len(g) < 3 * min_per_tercile:
            continue
        terc = pd.qcut(g['SIZE'].rank(method='first'), 3, labels=['small', 'mid', 'big'])
        for name in ['big', 'mid', 'small']:
            sub = g[terc == name]
            if len(sub) < min_per_tercile:
                continue
            dec = pd.qcut(sub[signal].rank(method='first'), 10, labels=False)
            ser[name][dt] = (sub.loc[dec==9, 'Fwd_Ret'].mean()
                             - sub.loc[dec==0, 'Fwd_Ret'].mean() - 2*cost_bps/1e4)
    ppy = 252.0 / H; out = {}
    for name in ['big', 'mid', 'small']:
        s = pd.Series(ser[name]).dropna()
        out[name] = dict(
            Sharpe=(s.mean()/s.std(ddof=1)*np.sqrt(ppy) if len(s) > 1 and s.std(ddof=1) > 0 else np.nan),
            N=len(s), medDV=None)
    return out

print("="*72)
print("PRE-REGISTERED SIZE-TERCILE TEST  (within S&P 500; smaller third of large caps)")
print("="*72)
print("H0: no size gradient.  H1: Sharpe rises big -> mid -> small.\n")

rows = []
for n, h in PREREG_CONFIGS:
    eb = _rebuild(n, h)                 # from the scientific-analysis cell; rebuilds eval_base from cache
    if eb is None:
        print(f"I{n}/R{h}: no cached predictions, skipping."); continue
    res  = size_tercile_ls(eb, df_market, 'CNN_Prob', h)
    resc = size_tercile_ls(eb, df_market, 'CNN_Prob', h, cost_bps=5)   # net-of-cost sanity
    big, mid, sml = res['big']['Sharpe'], res['mid']['Sharpe'], res['small']['Sharpe']
    mono = (sml > mid > big)
    print(f"I{n}/R{h}:")
    print(f"   gross  big {big:+.2f} (N={res['big']['N']})  mid {mid:+.2f}  small {sml:+.2f}"
          f"   -> {'MONOTONE small>mid>big' if mono else 'not monotone'}")
    print(f"   @5bps  big {resc['big']['Sharpe']:+.2f}  mid {resc['mid']['Sharpe']:+.2f}"
          f"  small {resc['small']['Sharpe']:+.2f}")
    print(f"   size gradient (small - big): {sml - big:+.2f}\n")
    for name, r in res.items():
        rows.append({'Config': f'I{n}/R{h}', 'Tercile': name,
                     'Sharpe_gross': r['Sharpe'], 'Sharpe_5bps': resc[name]['Sharpe'], 'N': r['N']})

pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, 'size_tercile_results.csv'), index=False)
print("="*72)
print("READ:")
print("  small > mid > big in BOTH configs  -> supports the size hypothesis; the effect")
print("     is a size gradient even within large caps. Report it (with the CRSP caveat).")
print("  flat / non-monotone                -> the effect is not a size story at S&P-500")
print("     scale; the earlier null stands and is not explained by size.")
print("  Note: even a positive gradient likely dies by 5bps -- small caps have wide")
print("     spreads, so 'statistically present' != 'tradeable'. State both.")
print(f"\nsaved -> {os.path.join(OUT_DIR, 'size_tercile_results.csv')}")

PRE-REGISTERED SIZE-TERCILE TEST  (within S&P 500; smaller third of large caps)
H0: no size gradient.  H1: Sharpe rises big -> mid -> small.

[00:17:20.2] N=5: 2,586,955 images on disk (1.2 GB), 2,586,955 usable | mem=rss 1.8G mps 0.0G
I5/R5:
   gross  big +0.76 (N=1081)  mid +0.48  small +0.03   -> not monotone
   @5bps  big +0.41  mid +0.11  small -0.21
   size gradient (small - big): -0.73

[00:17:39.4] N=20: 2,583,356 images on disk (9.9 GB), 2,583,356 usable | mem=rss 2.7G mps 0.0G
I20/R20:
   gross  big +0.27 (N=270)  mid +0.04  small +0.24   -> not monotone
   @5bps  big +0.18  mid -0.06  small +0.15
   size gradient (small - big): -0.04

READ:
  small > mid > big in BOTH configs  -> supports the size hypothesis; the effect
     is a size gradient even within large caps. Report it (with the CRSP caveat).
  flat / non-monotone                -> the effect is not a size story at S&P-500
     scale; the earlier null stands and is not explained by size.
  Note: even a positive gradi